# 05 - Binary Classification Training (2-Class)

Full training pipeline for binary ICH detection: Phase 1 classifier warmup + Phase 2 fine-tuning with MixUp/CutMix, EMA, RandAugment. Uses clean_training_data/.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
class NumpyDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = sorted([f for f in os.listdir(data_dir) if f.startswith("chunk_")])
        self.label_files = sorted([f for f in os.listdir(data_dir) if f.startswith("labels_chunk")])
        assert len(self.image_files) == len(self.label_files)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        imgs = np.load(os.path.join(self.data_dir, self.image_files[idx]), mmap_mode="r")
        lbls = np.load(os.path.join(self.data_dir, self.label_files[idx]), mmap_mode="r")
        return imgs, lbls


In [8]:
from torch.utils.data import DataLoader
import os 
train_dataset = NumpyDataset("train_data")  # same folder as training

train_loader = DataLoader(
    train_dataset,
    batch_size=1,                 # ❗ important
    shuffle=True,                 # OK for sampling
    collate_fn=lambda x: x[0],    # same as training
    num_workers=2
)


In [ ]:
# ---------------------------
# ConvNeXt Large — Full Training Script
# Phase1 (classifier warmup) + Phase2 (fine-tune)
# RandAugment (PIL) + stronger MixUp/CutMix + GPU EMA
# ---------------------------

import os
import copy
import numpy as np
from PIL import Image
from contextlib import contextmanager
import itertools
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms.functional as TF
from torchvision.transforms import RandAugment

# ---------------------------
# Device / speed options
# ---------------------------
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------------------
# NOTE: define your ConvNeXt model before running this script
# Example (user must provide): 
# from torchvision.models import convnext_large, ConvNeXt_Large_Weights
# model = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)
# Then paste this script and run.
# ---------------------------
model = model.to(device)  # assume 'model' exists

# ---------------------------
# Dataset classes (chunked .npy files)
# ---------------------------
class NumpyDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = sorted([f for f in os.listdir(data_dir) if f.startswith("chunk_")])
        self.label_files = sorted([f for f in os.listdir(data_dir) if f.startswith("labels_chunk")])
        assert len(self.image_files) == len(self.label_files), "chunk/label count mismatch"

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        imgs = np.load(os.path.join(self.data_dir, self.image_files[idx]), mmap_mode="r")
        lbls = np.load(os.path.join(self.data_dir, self.label_files[idx]), mmap_mode="r")
        return imgs, lbls


class ValidationDataset(Dataset):
    def __init__(self, imgs_path, labels_path):
        self.images = np.load(imgs_path, mmap_mode="r")
        self.labels = np.load(labels_path, mmap_mode="r")
        assert len(self.images) == len(self.labels)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx].copy()).float()  # H,W,3 or similar
        lbl = int(self.labels[idx])
        return img, lbl

# ---------------------------
# RandAugment (PIL) setup
# ---------------------------
rand_aug = RandAugment(num_ops=2, magnitude=9)

def apply_randaugment(batch):
    """
    Input: batch tensor C,H,W on GPU (or CPU). Returns tensor C,H,W on device.
    We handle float ranges by mapping to uint8 before PIL.
    """
    out = []
    for img in batch:  # img shape C,H,W
        arr = img.permute(1,2,0).cpu().numpy()
        # handle float [0,1] or [0,255]
        if arr.max() <= 1.0:
            arr = (arr * 255.0).clip(0,255).astype(np.uint8)
        else:
            arr = arr.clip(0,255).astype(np.uint8)
        pil = Image.fromarray(arr)
        pil_aug = rand_aug(pil)
        ten = TF.to_tensor(pil_aug).to(device)
        out.append(ten)
    return torch.stack(out, dim=0)

# ---------------------------
# Normalization constants
# ---------------------------
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

# ---------------------------
# Augment pipelines
# ---------------------------
def augment_phase1(batch):
    # light augment for Phase 1 (classifier warmup)
    # batch: tensor (B,C,H,W) values usually 0-255 or 0-1
    if batch.max() > 2.0:
        batch = batch / 255.0
    if torch.rand(1) < 0.5:
        batch = batch.flip(-1)
    batch = F.interpolate(batch, size=(256,256), mode='bilinear', align_corners=False)
    batch = (batch - mean) / std
    return batch

def augment_phase2_balanced(batch):
    # batch input: float tensor in 0-255 or 0-1
    if batch.max() > 2.0:
        batch = batch / 255.0
        batch = (batch * 255.0).clamp(0,255)

    # 1/3 chance → heavy RandAugment (slow)
    if torch.rand(1) < 0.33:
        batch = apply_randaugment(batch)
    else:
        # Fast augment (light)
        if torch.rand(1) < 0.5:
            batch = batch.flip(-1)
        if torch.rand(1) < 0.3:
            batch = batch.flip(-2)

        # Small rotation
        if torch.rand(1) < 0.5:
            angle = (torch.rand(1).item() * 20.0 - 10.0)
            batch = TF.rotate(batch, angle, interpolation=TF.InterpolationMode.BILINEAR)

        # Light color jitter
        if torch.rand(1) < 0.5:
            batch = TF.adjust_brightness(batch, 1.0 + 0.1 * torch.randn(1).item())
            batch = TF.adjust_contrast(batch, 1.0 + 0.1 * torch.randn(1).item())

    # Resize + normalize
    batch = F.interpolate(batch, size=(256,256), mode='bilinear', align_corners=False)
    batch = (batch - mean) / std
    return batch

# ---------------------------
# MixUp / CutMix (stronger mixups)
# ---------------------------
def mixup_data(x, y, alpha=0.6):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.size()
    idx = torch.randperm(B).to(x.device)

    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)

    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1. - ((x2 - x1) * (y2 - y1) / (W * H))
    return x, y, y[idx], lam

def mixup_criterion(crit, pred, y_a, y_b, lam):
    return lam * crit(pred, y_a) + (1 - lam) * crit(pred, y_b)

# ---------------------------
# GPU-safe EMA
# ---------------------------
class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        esd = self.ema.state_dict()
        for k in msd.keys():
            esd[k].mul_(self.decay).add_(msd[k], alpha=1.0 - self.decay)

# ---------------------------
# Validation (uses ema.ema if provided)
# ---------------------------
@torch.no_grad()
def validate_model(model, dataset, criterion, batch_size=64, ema: ModelEMA = None):
    eval_model = ema.ema if ema is not None else model
    eval_model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for start in tqdm(range(0, len(dataset), batch_size), desc="Validation", ncols=100):
        end = min(start + batch_size, len(dataset))
        imgs = torch.from_numpy(dataset.images[start:end].copy()).float().to(device)  # (B,H,W,3)
        labels = torch.tensor(dataset.labels[start:end]).long().to(device)

        imgs = imgs.permute(0,3,1,2)  # -> (B,C,H,W)
        if imgs.max() > 2.0:
            imgs = imgs / 255.0
        imgs = F.interpolate(imgs, size=(256,256), mode='bilinear', align_corners=False)
        imgs = (imgs - mean) / std

        with torch.amp.autocast("cuda"):
            out = eval_model(imgs)
            loss = criterion(out, labels)

        running_loss += loss.item() * labels.size(0)
        _, preds = out.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total

# ---------------------------
# Config + Data loaders
# ---------------------------
train_dir = "train_data"
val_images = "val_data.npy"
val_labels = "val_label.npy"

train_dataset = NumpyDataset("train_data_clean")
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=lambda x: x[0])

val_dataset = ValidationDataset(val_images, val_labels)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 2.5]).to(device),
    label_smoothing=0.05
)

scaler = torch.cuda.amp.GradScaler()

CHUNK = 32                # mini-batch inside chunk
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 15
PATIENCE = 7

# ---------------------------
# PHASE 1: Train classifier head only
# ---------------------------
print("\n===== PHASE 1: Classifier warmup =====\n")

# Freeze all backbone params, unfreeze classifier
for p in model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE1_EPOCHS)

for ep in range(PHASE1_EPOCHS):
    model.train()
    total, correct, loss_sum = 0, 0, 0

    for imgs, lbls in tqdm(train_loader, desc=f"P1 Epoch {ep+1}/{PHASE1_EPOCHS}", ncols=100):
        # imgs: numpy array (N, H, W, C), lbls: numpy array (N,1) or (N,)
        for i in range(0, len(imgs), CHUNK):
            batch = torch.from_numpy(imgs[i:i+CHUNK].copy()).float().permute(0,3,1,2).to(device)
            labels = torch.tensor(lbls[i:i+CHUNK]).long().squeeze(-1).to(device) if (np.array(lbls).ndim>1) else torch.tensor(lbls[i:i+CHUNK]).long().to(device)

            batch = augment_phase1(batch)

            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(batch)
                loss = criterion(preds, labels)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            _, pcls = preds.max(1)
            correct += (pcls == labels).sum().item()
            total += labels.size(0)
            loss_sum += loss.item() * labels.size(0)

    sched.step()
    train_acc = 100.0 * correct / total if total > 0 else 0.0
    val_loss, val_acc = validate_model(model, val_dataset, criterion)
    print(f"P1 Epoch {ep+1}: Train={train_acc:.2f}% | Val={val_acc:.2f}%")

# ---------------------------
# PHASE 2: Fine-tune with heavy augment + mixup/cutmix
# ---------------------------
print("\n===== PHASE 2: Fine-tuning =====\n")

# Freeze early stages 0,1,2
for name, p in model.named_parameters():
    if ("stages.0" in name) or ("stages.1" in name) or ("stages.2" in name):
        p.requires_grad = False
    else:
        p.requires_grad = True

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE2_EPOCHS)
ema = ModelEMA(model)

best_val = 0.0
no_improve = 0

for ep in range(PHASE2_EPOCHS):
    model.train()
    total, correct, loss_sum = 0, 0, 0

    for imgs, lbls in tqdm(train_loader, desc=f"P2 Epoch {ep+1}/{PHASE2_EPOCHS}", ncols=100):
        for i in range(0, len(imgs), CHUNK):
            batch = torch.from_numpy(imgs[i:i+CHUNK].copy()).float().permute(0,3,1,2).to(device)
            labels_np = lbls[i:i+CHUNK]
            # handle various label shapes
            if isinstance(labels_np, np.ndarray) and labels_np.ndim > 1:
                labels = torch.tensor(np.array(labels_np).squeeze(-1)).long().to(device)
            else:
                labels = torch.tensor(labels_np).long().to(device)

            batch = augment_phase2_balanced(batch)


            # stronger MixUp / CutMix
            if np.random.rand() < 0.5:
                batch, y1, y2, lam = mixup_data(batch, labels, alpha=0.6)
            else:
                batch, y1, y2, lam = cutmix_data(batch, labels, alpha=1.0)

            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(batch)
                loss = mixup_criterion(criterion, preds, y1, y2, lam)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            # update EMA (GPU-only)
            ema.update(model)

            _, pcls = preds.max(1)
            correct += (pcls == labels).sum().item()
            total += labels.size(0)
            loss_sum += loss.item() * labels.size(0)

    sched.step()

    train_acc = 100.0 * correct / total if total > 0 else 0.0
    val_loss, val_acc = validate_model(model, val_dataset, criterion, ema=ema)
    print(f"P2 Epoch {ep+1}: Train={train_acc:.2f}% | Val={val_acc:.2f}%")

    if val_acc > best_val:
        best_val = val_acc
        no_improve = 0
        torch.save(model.state_dict(), "best_convnext_large(removing_outlier.pth")
        print("✔ Saved best model")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("⛔ Early stopping triggered")
            break

print("\n🎉 Training finished. Best Val Acc: {:.2f}% 🎉".format(best_val))


/tmp/ipykernel_8437/1304937008.py:251: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Device: cuda

===== PHASE 1: Classifier warmup =====





alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:27<00:00,  8.42it/s]

P1 Epoch 1: Train=93.04% | Val=92.79%




alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:20<00:00, 11.46it/s]

...



alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:19<00:00, 11.97it/s]

P2 Epoch 8: Train=73.97% | Val=92.37%




alidation: 100%|█████████████████████████████████████████████████| 235/235 [00:19<00:00, 11.76it/s]

P2 Epoch 9: Train=73.93% | Val=92.44%


P2 Epoch 10/15:   0%|                                                        | 0/12 [00:00<?, ?it/s]